# Clearview AI: When Nobody Is Ever Asked to Enroll

In this notebook I look at Clearview AI as a seventh mechanism behind an AI ethics failure, and a genuinely different one from everything else in this series so far.

Every one of the first six notebooks was about an unequal outcome, some score, threshold, or decision that treated groups of people differently once they were already inside a system. Clearview AI raises a different question entirely: not whether a system treats two groups unequally, but whether anyone ever agreed to be part of the system at all, at any scale, for any purpose.

In this notebook, I will:

- Summarize what Clearview AI actually did, and what investigations and regulators found
- Simulate how many people end up enrolled in a bounded, consent-based system versus a scraped one
- Test the fix Clearview's own defense pointed to, respecting each platform's stated terms, and see how little it actually changes
- Test what a deletion or opt-out request actually removes, once a face has already been converted into a searchable biometric template
- Find the fix that actually reduced real-world harm, and see why it came from outside the system entirely, not from a change to the model or the data pipeline

## 1. Background: What Was Reported

Starting around 2017, Clearview AI, a small startup founded by Hoan Ton-That, scraped several billion photographs from public web pages and social media platforms, including Facebook, Instagram, Twitter, LinkedIn, and YouTube, without asking those platforms or the people in the photos for permission. It used the photos to build a facial recognition search tool: upload any photo of a face, and the tool returns links to every other photo of that same face it has found on the open web, along with wherever those photos point, a social media profile, a name, a location tag.

A 2020 New York Times investigation revealed the tool existed and that it was already in use by hundreds of law enforcement agencies, mostly without any public disclosure, procurement review, or department policy governing its use. A later BuzzFeed News investigation found that access had also been given, informally and outside the company's own stated policy of selling only to law enforcement, to a range of private companies, individual investors, and people running personal searches unrelated to any investigation. Facebook, Twitter, YouTube, and LinkedIn sent cease-and-desist letters demanding Clearview stop scraping their platforms and delete the data, which by the company's own admission did not fully happen. Illinois residents sued under the state's Biometric Information Privacy Act, a rare US law requiring informed consent before a company collects someone's biometric data; the case settled in 2022 with Clearview agreeing to stop selling access to most private companies and individuals within the United States, while continuing to sell to law enforcement and government contractors. Separately, data protection regulators in Canada, the United Kingdom, France, Italy, and Australia each ruled that Clearview had violated their countries' privacy laws by collecting citizens' biometric data without consent, and ordered it to delete that data and stop collecting more.

## 2. Why This Is a Different Mechanism

Every one of the first six notebooks in this series measured what happened to people already inside a system, comparing an outcome, a score, a flag, a credit limit, between groups. Clearview AI raises a prior question: how does anyone end up inside the system at all?

A mugshot database, whatever its other problems, only includes people who were arrested, a bounded, legally-defined, documented event. Clearview's database includes anyone who has ever had a photo of their face posted publicly online, by themselves or by someone else, which by now is most of the population of any country with widespread internet access, none of whom were arrested, asked, or even told. There is no threshold to tune and no proxy feature to remove here, because what this notebook has to measure isn't a rate at which a decision goes one way or another, it's how many people were ever given a choice about being enrolled in a permanent, searchable biometric system in the first place, and what, if anything, happens when they later say no.

## 3. A Note on the Simulation Below

This notebook does not use Clearview's real numbers, which the company has never fully disclosed, and its billions of scraped photos aren't something I can or should try to reproduce. I use small, made-up populations to make two specific, well-documented mechanisms concrete: how many people end up enrolled without ever being asked, and how little a later opt-out or deletion request actually removes once a face has already been converted into a stored biometric template.

## 4. Simulating Who Ends Up Enrolled

I generate a population of simulated people. A small share of them have ever been arrested, the only route into a traditional mugshot database. A much larger share have at some point had a public photo of their face posted online, by themselves or by someone tagging them, the only requirement to end up in a scraped database like Clearview's.

In [ ]:
import random

random.seed(4)

N_PEOPLE = 10000
ARREST_RATE = 0.03
PUBLIC_PHOTO_RATE = 0.85


def make_population(n, arrest_rate, public_photo_rate):
    """Returns a list of person dicts with an arrest history flag and a public photo flag."""
    people = []
    for _ in range(n):
        people.append({
            "was_arrested": random.random() < arrest_rate,
            "has_public_photo": random.random() < public_photo_rate,
        })
    return people


population = make_population(N_PEOPLE, ARREST_RATE, PUBLIC_PHOTO_RATE)

## 5. Measuring Enrollment in Each Kind of System

I compare how many people end up enrolled in each kind of system: the mugshot database only enrolls people with an arrest on record, the scraped database enrolls anyone with a public photo, regardless of any wrongdoing.

In [ ]:
def enrollment_rate(population, key):
    """Returns the fraction of a population enrolled in a system keyed on the given flag."""
    enrolled = [p for p in population if p[key]]
    return len(enrolled) / len(population)


mugshot_rate = enrollment_rate(population, "was_arrested")
scraped_rate = enrollment_rate(population, "has_public_photo")

print("Enrolled in a traditional mugshot database:", round(mugshot_rate, 3))
print("Enrolled in a Clearview-style scraped database:", round(scraped_rate, 3))

## 6. Checking for Any Documented Process Behind Enrollment

Both numbers above describe enrollment, but not consent, an arrest is a legal process, not a request for permission. So I check how many enrolled people in each system went through any documented, rule-bound process before being enrolled, versus simply having existed on the internet.

In [ ]:
def documented_process_rate(population, key):
    """Returns the fraction of a system's enrolled people who went through a documented process to get there."""
    enrolled = [p for p in population if p[key]]
    if not enrolled:
        return 0.0
    documented = [p for p in enrolled if key == "was_arrested"]
    return len(documented) / len(enrolled)


print(
    "Enrolled people who went through a documented process, mugshot database:",
    round(documented_process_rate(population, "was_arrested"), 3),
)
print(
    "Enrolled people who went through a documented process, scraped database:",
    round(documented_process_rate(population, "has_public_photo"), 3),
)